<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 9B: *Fire Spread Feature Ablation*
##### Version Number: 4.0
---
### Contents  
> 1. **
> 2. **
> 3. **
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_spread = pd.read_csv('../data/processed/X_spread.csv')
y_spread = pd.read_csv('../data/processed/y_spread.csv').squeeze()  # Load as Series
details_spread = pd.read_csv('../data/processed/details_spread.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/spread_best_strategy.csv')

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)
    
with open('feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_spread,y_spread], axis=1)
subset = subset_df(reform, 'Target_Spread', 500)

y = subset['Target_Spread']
X = subset.drop(columns='Target_Spread')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build tuned models
spread_xgb = xgb.XGBClassifier(**XGB_parameters)
spread_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 150,
 'max_depth': 15,
 'min_samples_split': 2,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 3,
 'n_estimators': 100,
 'max_depth': 4,
 'learning_rate': 0.1,
 'verbosity': 0}

In [7]:
models = {
    "XGB": spread_xgb, 
    "RF": spread_rf,
}

## SHAP

In [8]:
xgb_top_10 = get_shap(spread_xgb, X_xgb, y_xgb)
xgb_top_10

,0,1
0,influence_zone,0.201566
1,reservoir_count,0.187492
2,Palmer Drought Severity Index,0.183533
3,road_length_meters,0.153164
4,1000-hour Dead Fuel Moisture,0.136745
5,Solar Radiation 7 Day Avg,0.136122
6,Daily Maximum Air Temperature 7 Day Avg,0.131018
7,Daily Minimum Air Temperature,0.130149
8,Precipitation 7 Day Avg,0.091849
9,elevation_range,0.090969


In [9]:
rf_top_10 = get_shap(spread_rf, X_rf, y_rf)
rf_top_10 

 43%|=========           | 385/900 [00:11<00:14]       

 46%|=========           | 415/900 [00:12<00:14]       

 49%|==========          | 445/900 [00:13<00:13]       

 53%|===========         | 477/900 [00:14<00:12]       

 56%|===========         | 507/900 [00:15<00:11]       

...

 86%|=================   | 773/900 [00:22<00:03]       

 90%|==================  | 813/900 [00:23<00:02]       

 95%|=================== | 853/900 [00:24<00:01]       

 99%|===================| 887/900 [00:25<00:00]       

,0,1
0,influence_zone,0.014975
1,power_line_density_x_total_housing,0.012678
2,Daily Minimum Air Temperature,0.012014
3,total_housing,0.011920
4,road_length_meters,0.011446
5,road_density,0.011411
6,intermix_zone,0.010241
7,Vapor Pressure Deficit 7 Day Avg,0.010238
8,total_population,0.009359
9,power_line_meters,0.008913


## Ablation

In [10]:
for set_name, columns in feature_sets.items():
    print(f"{set_name}: {columns}")
    print("\n")

Water Demand: ['Actual Evapotranspiration', 'Solar Radiation', 'Solar Radiation 7 Day Avg', 'Daily Minimum Air Temperature', 'Daily Maximum Air Temperature', 'Daily Maximum Air Temperature 7 Day Avg', 'Daily Minimum Air Temperature 7 Day Avg', 'Vapor Pressure Deficit', 'Vapor Pressure Deficit 7 Day Avg', 'Wind Speed', 'Wind Speed 7 Day Avg']


Water Supply: ['Precipitation', 'Precipitation 7 Day Avg', 'Maximum Relative Humidity', 'Minimum Relative Humidity', 'Maximum Relative Humidity 7 Day Avg', 'Minimum Relative Humidity 7 Day Avg', 'Specific Humidity', '100-hour Dead Fuel Moisture', '1000-hour Dead Fuel Moisture']


Water Supply Indexes: ['Standardized Precipitation Index 30-Day', 'Standardized Precipitation Index 180-Day', 'Standardized Precipitation Evapotranspiration Index 30-Day', 'Standardized Precipitation Evapotranspiration Index 90-Day', 'Standardized Precipitation Evapotranspiration Index 180-Day', 'Palmer Drought Severity Index']


Fire Danger: ['Burning Index', 'Energy Re

In [11]:
compare = []

for set_name, set_list in feature_sets.items():
    for model_name, model in models.items():
        print("Testing " + f"{model_name}: {set_name}")
        X_section = X.drop(columns = set_list)
        compare.append(compare_model(model, X_section, y, best_strategy,
                                     name = model_name, test_set = set_name))

Testing XGB: Water Demand


Testing RF: Water Demand


Testing XGB: Water Supply


Testing RF: Water Supply


Testing XGB: Water Supply Indexes


...

Testing RF: Interactions


Testing XGB: Wind Slope


Testing RF: Wind Slope


Testing XGB: Others


Testing RF: Others


In [12]:
comparisons = pd.DataFrame(compare)
comparisons

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall
0,Water Demand,XGB,0.464916,0.475189,0.491071
1,Water Demand,RF,0.483351,0.493041,0.482143
2,Water Supply,XGB,0.502058,0.512356,0.508929
3,Water Supply,RF,0.479642,0.491478,0.491071
4,Water Supply Indexes,XGB,0.513837,0.520858,0.562500
5,Water Supply Indexes,RF,0.519956,0.529814,0.535714
6,Fire Danger,XGB,0.505624,0.512481,0.535714
7,Fire Danger,RF,0.486587,0.498105,0.500000
8,Social,XGB,0.525853,0.530899,0.544643
9,Social,RF,0.503701,0.513555,0.517857
